# Capstone 3: Insurance Fraud Detection

**Business Problem:** An insurance company suspects that approximately 15% of their claims are fraudulent. They want you to build a machine learning model that can flag potentially fraudulent policyholders.

**Database:** `insurance_fraud.db` — 4 tables that you must JOIN and engineer features from.

**The Rules:** This is your Month 1 capstone. You must write ALL code from memory. No Cheat Sheet. I will review your work as your Senior Developer.

---


## Step 1: Database Exploration & SQL Extraction (10 minutes)
1. Connect to `insurance_fraud.db`.
2. Explore all 4 tables — understand what each one contains and how they are linked.
3. Write a SQL query that JOINs `policyholders`, `agents`, and `fraud_labels`.
4. Load into a DataFrame. Print `df.head()`, `df.info()`, and the percentage of missing values.
5. In a comment, note which columns have missing values and which have outliers.

In [48]:
# Step 1 Code Here
import sqlite3
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import numpy as np

conn = sqlite3.connect('insurance_fraud.db')

query = """
SELECT * FROM agents
LEFT JOIN policyholders ON agents.agent_id = policyholders.agent_id
LEFT JOIN fraud_labels ON policyholders.policyholder_id = fraud_labels.policyholder_id
LEFT JOIN claims ON policyholders.policyholder_id = claims.policyholder_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,agent_id,agent_name,agency_type,years_experience,policyholder_id,agent_id,age,gender,bmi,smoker,region,income_bracket,policyholder_id,is_fraudulent,claim_id,policyholder_id,claim_amount,claim_type,hospitalization_days,treatment_type
0,1,Agent_A,Corporate,21,39,1,67,Female,NaN,Yes,East,Very High,39,0,19.0,39.0,1466.61,Prescription,16.0,Outpatient
1,1,Agent_A,Corporate,21,39,1,67,Female,NaN,Yes,East,Very High,39,0,191.0,39.0,2720.18,Medical,23.0,Rehabilitation
2,1,Agent_A,Corporate,21,39,1,67,Female,NaN,Yes,East,Very High,39,0,235.0,39.0,1950.38,Surgical,27.0,Outpatient
3,1,Agent_A,Corporate,21,39,1,67,Female,NaN,Yes,East,Very High,39,0,805.0,39.0,2864.25,Emergency,6.0,Outpatient
4,1,Agent_A,Corporate,21,39,1,67,Female,NaN,Yes,East,Very High,39,0,1029.0,39.0,7139.77,Medical,7.0,Rehabilitation


## Step 2: SQL Feature Engineering (15 minutes)
Go BACK to your SQL query and upgrade it. Using CTEs, create these features for each policyholder:

1. `total_claims` — Total number of claims filed.
2. `total_claim_amount` — Sum of all claim amounts.
3. `avg_claim_amount` — Average claim amount.
4. `max_hospitalization` — Maximum hospitalization days across all claims.
5. `emergency_count` — Number of Emergency claims (using `CASE WHEN` or a filtered COUNT).
6. `claim_type_variety` — Number of distinct claim types used (using `COUNT(DISTINCT ...)`)

Also add a Window Function:
7. `avg_region_claims` — Average total_claim_amount for policyholders in the same region.

Reload the upgraded DataFrame. Print `df.shape` and `df.head()`.

**Business Insight:** Why are these features useful for detecting fraud? Write your reasoning in a comment.

In [49]:
# Step 2 Code Here
query = """
WITH ClaimStats AS (
    SELECT 
        policyholder_id,
        COUNT(claim_id) AS total_claims, 
        SUM(claim_amount) AS total_claim_amount,
        AVG(claim_amount) AS avg_claim_amount,
        SUM(CASE WHEN claim_type = 'Emergency' THEN 1 ElSE 0 END) AS emergency_count,
        MAX(hospitalization_days) AS max_hospitalization,
        COUNT(DISTINCT claim_type) AS claim_type_variety
    FROM claims
    GROUP BY policyholder_id
)
SELECT 
    policyholders.*,
    agents.agent_name,
    agents.agency_type,
    agents.years_experience,
    c.total_claims,
    c.total_claim_amount,
    c.avg_claim_amount,
    c.emergency_count,
    c.max_hospitalization,
    c.claim_type_variety,
    AVG(c.total_claim_amount) OVER (PARTITION BY region) AS avg_region_claims,
    f.is_fraudulent
FROM policyholders
LEFT JOIN ClaimStats c ON c.policyholder_id = policyholders.policyholder_id
LEFT JOIN fraud_labels f ON f.policyholder_id = policyholders.policyholder_id
LEFT JOIN agents ON agents.agent_id = policyholders.agent_id
ORDER BY policyholders.policyholder_id
"""
df = pd.read_sql_query(query, conn)
df = df.fillna(np.nan)
df.head()

#When we are talking about Health Insurance understanding the background of an individual gives us more clue as to whether the claim is true or fraudulent
#Example we have a group for Smokers, people with a history of smoking are most likely to have true claims depending on the illness.
#Income bracket is also a determining factor. People from low income bracket are more likely to commit insurance fraud to save money.

,policyholder_id,agent_id,age,gender,bmi,smoker,region,income_bracket,agent_name,agency_type,years_experience,total_claims,total_claim_amount,avg_claim_amount,emergency_count,max_hospitalization,claim_type_variety,avg_region_claims,is_fraudulent
0,1,14,49,Male,26.8,Yes,West,High,Agent_N,Corporate,16,2.0,16983.74,8491.870000,0.0,23.0,1.0,32369.299338,0
1,2,14,18,Male,31.1,Yes,South,High,Agent_N,Corporate,16,3.0,32510.05,10836.683333,1.0,24.0,3.0,41544.583359,1
2,3,15,22,Female,27.2,No,West,Medium,Agent_O,Independent,15,1.0,1874.99,1874.990000,0.0,11.0,1.0,32369.299338,0
3,4,14,62,Female,NaN,No,East,Very High,Agent_N,Corporate,16,NaN,NaN,NaN,NaN,NaN,NaN,37509.523542,0
4,5,3,21,Female,32.6,No,East,Low,Agent_C,Corporate,24,NaN,NaN,NaN,NaN,NaN,NaN,37509.523542,0


## Step 3: Preprocessing Pipeline (15 minutes)
Build a complete `ColumnTransformer` with `Pipeline` steps.

For EACH column, you must:
1. Choose the correct `SimpleImputer` strategy (and justify WHY in a comment).
2. Choose the correct Scaler or Encoder (and justify WHY in a comment).

**Think carefully about:**
- `income_bracket` has a natural ORDER (Low < Medium < High < Very High). Which encoder?
- `bmi` and `claim_amount` have extreme outliers. Which scaler?
- `smoker` has NULLs. What fill strategy makes business sense?

Drop any useless columns (IDs, names). Split into X and y. `fit_transform` training data, `transform` test data.

In [50]:
df.isna().mean()

policyholder_id        0.000000
agent_id               0.000000
age                    0.000000
gender                 0.000000
bmi                    0.131667
smoker                 0.075000
region                 0.000000
income_bracket         0.036667
agent_name             0.000000
agency_type            0.000000
years_experience       0.000000
total_claims           0.081667
total_claim_amount     0.081667
avg_claim_amount       0.081667
emergency_count        0.081667
max_hospitalization    0.118333
claim_type_variety     0.081667
avg_region_claims      0.000000
is_fraudulent          0.000000
dtype: float64

In [51]:
df['income_bracket'].unique()

array(['High', 'Medium', 'Very High', 'Low', nan], dtype=object)

In [56]:
df['total_claim_amount'].max()

544121.47

In [91]:
df['avg_claim_amount'].describe()

count       551.000000
mean      12585.995339
std       30203.393947
min         416.330000
25%        4015.821667
50%        6450.076667
75%       10351.668333
max      405030.260000
Name: avg_claim_amount, dtype: float64

In [98]:
# Step 3 Code Here
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, learning_curve

X = df.drop(columns=['agent_id', 'policyholder_id', 'agent_name', 'is_fraudulent'])
y = df['is_fraudulent']
num_cols = X.select_dtypes(['int64', 'float64']).columns.tolist()
num_cols.remove('total_claim_amount')
num_cols.remove('avg_claim_amount')
num_cols.remove('emergency_count')
cat_cols = ['gender', 'region', 'agency_type']
smoke_cols = ['smoker']
income_cols = ['income_bracket']
outlier_cols = ['total_claim_amount', 'avg_claim_amount']
emerg_cols = ['emergency_count']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoding', OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']]))
])

outlier_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', RobustScaler())
])

smoke_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

emerg_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

processor = ColumnTransformer ([
    ('nums', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    ('ord', ord_pipeline, income_cols),
    ('outlier', outlier_pipeline, outlier_cols),
    ('smoke', smoke_pipeline, smoke_cols),
    ('emerg', emerg_pipeline, emerg_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Step 4: Model Training & Tuning (10 minutes)
1. Train a baseline `XGBClassifier` and print accuracy.
2. Use `RandomizedSearchCV` with at least 3 hyperparameters and `cv=5`.
3. Print the best parameters.
4. Print the best model's accuracy on the test set.

In [99]:
# Step 4 Code Here
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

full_pipeline = Pipeline([
    ('preprocessor', processor),
    ('model', XGBClassifier())
])
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 8],
    'model__learning_rate': [0.01, 0.05, 0.1]
}
random_search = RandomizedSearchCV(full_pipeline, param_grid, cv=5)

random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"The testing accuracy is: {accuracy}")
print(random_search.best_score_)

The testing accuracy is: 0.85
0.85


In [100]:
sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train,
    train_sizes= np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring= 'accuracy',
    random_state= 42
)

best_model_diagnosis = pd.DataFrame({
    'Train Size': sizes,
    'CV Train Mean': np.mean(np.round(train_scores, 4), axis=1),
    'CV Val Mean': np.mean(np.round(val_scores, 4), axis=1),
    'CV Gap': np.round((np.mean(train_scores, axis=1) - np.mean(val_scores, axis=1)),4)
})



In [101]:
print(best_model_diagnosis.to_string(index=False))

 Train Size  CV Train Mean  CV Val Mean  CV Gap
         38        0.77372      0.81670 -0.0430
         76        0.82632      0.83544 -0.0091
        115        0.82958      0.85004 -0.0204
        153        0.83794      0.85004 -0.0121
        192        0.85212      0.85004  0.0021
        230        0.85392      0.85004  0.0039
        268        0.84256      0.85004 -0.0075
        307        0.84558      0.85004 -0.0044
        345        0.84756      0.85004 -0.0025
        384        0.85004      0.85004  0.0000


## Step 5: Feature Selection (10 minutes)
1. Print `best_model.feature_importances_` matched with feature names.
2. Identify the top 5 most important features.
3. Identify any features with near-zero importance.
4. In a comment: Which of your SQL-engineered features was most valuable? Would you drop any features and retrain?

In [102]:
# Step 5 Code Here
xgb_model = best_model['model']
features = best_model['preprocessor'].get_feature_names_out()
feature_scores = xgb_model.feature_importances_

features_review = pd.DataFrame({
    "Features": features,
    'Score': feature_scores
}).sort_values('Score', ascending=False)

print(features_review.to_string(index=False))
#The Western region had the most impact on the model

                    Features    Score
 outlier__total_claim_amount 0.095412
      emerg__emergency_count 0.085406
                   nums__age 0.082350
  cat__agency_type_Exclusive 0.073115
                   nums__bmi 0.070149
         ord__income_bracket 0.069823
          nums__total_claims 0.066035
   nums__max_hospitalization 0.062539
            cat__region_West 0.060936
   outlier__avg_claim_amount 0.059753
           smoke__smoker_Yes 0.053002
cat__agency_type_Independent 0.051717
      nums__years_experience 0.045343
     nums__avg_region_claims 0.040840
            cat__gender_Male 0.036882
    nums__claim_type_variety 0.035950
       smoke__smoker_Unknown 0.010748
           cat__region_North 0.000000
           cat__region_South 0.000000


In [83]:
X_train['agency_type'].value_counts()

agency_type
Corporate      314
Exclusive      113
Independent     53
Name: count, dtype: int64

## Step 6: Model Diagnosis (10 minutes)
1. Compare Training Accuracy vs Testing Accuracy for your best model. Is it overfitting?
2. BONUS: Generate a learning curve for your best model. Does it need more data or a better algorithm?
3. In a comment, write your final diagnosis.

In [84]:
# Step 6 Code Here
#1. No the model is not overfitting. The training accuracy is 86% and the testing accuracy is 80.8%. This is not a sign of overfitting in the model.
#2. No we do not need more data for the model as the validation score is almost the same as the training score and the gap is minimal
#3. My diagnosis is that it is a good model. What we need are better engineered features. I also need to drop the features regarding the type of agency as that has no impact in the model.

## Step 7: The Business Report (10 minutes)
Answer ALL of these in comments:

1. The company currently catches 0% of fraud. Your model catches ~X%. How much money could this save if the average fraudulent claim is ₦5,000,000?
2. A fraud investigator can only review 50 cases per month. How would you use your model's predictions to prioritize which cases to investigate first?
3. If your model flags an innocent policyholder as fraudulent, what is the business consequence? Is it better to have MORE false alarms or FEWER? Why?
4. The CEO asks: "Can we just flag everyone with a claim above ₦300,000 as fraudulent?" Why is this a bad idea? Use your feature importance results to support your answer.

In [ ]:
# Step 7 Answers Here
#1. The accuracy of my model is 80%. At this rate the average fraudulent claim will reduce to by ~4,000,000.
#2. We rank and see who is more is likely to commit fraud and then hand the investigator the top 50 highest-confidence cases to audit
#3. It is better to have MORE false alarms. The point of the model is to detect fraud. The company is experiencing losses from fraudulent claims and we are trying to fix it, with better information from the policyholder there will be a chance for the status to be changed from fraudulent
#4. No. I dont really have an answer for this. Explain

In [88]:
X_train['avg_claim_amount'].describe()

count       440.000000
mean      12681.534025
std       29600.581118
min         416.330000
25%        4171.549792
50%        6507.024000
75%       10852.795000
max      405030.260000
Name: avg_claim_amount, dtype: float64

In [103]:
print(best_model.predict_proba(X_test))

[[0.77608246 0.22391756]
 [0.7952944  0.20470557]
 [0.92193365 0.07806633]
 [0.69587135 0.30412868]
 [0.66335297 0.336647  ]
 [0.8665978  0.13340224]
 [0.7842323  0.2157677 ]
 [0.7725389  0.2274611 ]
 [0.7702103  0.22978967]
 [0.9300096  0.0699904 ]
 [0.920736   0.07926397]
 [0.9138876  0.0861124 ]
 [0.8935259  0.10647412]
 [0.7881564  0.21184364]
 [0.83377385 0.16622613]
 [0.7938492  0.20615081]
 [0.8177124  0.18228759]
 [0.912365   0.08763498]
 [0.8854762  0.11452385]
 [0.90842426 0.09157576]
 [0.7600002  0.23999979]
 [0.6719266  0.3280734 ]
 [0.79446507 0.20553493]
 [0.7042959  0.29570416]
 [0.74219155 0.25780845]
 [0.90065634 0.09934364]
 [0.8435749  0.15642509]
 [0.9024311  0.09756892]
 [0.9148661  0.08513389]
 [0.7360946  0.2639054 ]
 [0.8806218  0.11937822]
 [0.9269779  0.07302216]
 [0.88941914 0.11058087]
 [0.8838669  0.11613306]
 [0.8072833  0.19271672]
 [0.839467   0.16053303]
 [0.852368   0.14763202]
 [0.87812835 0.12187165]
 [0.879731   0.12026902]
 [0.8399915  0.16000849]
